# Setup

## Making conda env viewable to juypter notebook
1. Go to the OnDemand dashboard, click files and then home directory 
2. Open Terminal 
3. Run the following commands

    a. `module load anaconda3/2023.03`
    
    b. `conda env list` (you should see anphy_sleep_env, if not see pace_anphy_instructions.pdf)
    
    c. `conda activate anphy_sleep_env`
    
    d. `conda install ipykernel -y`
    
    e. `python -m ipykernel install --user --name anphy_sleep_env`    
    
4. Now, `anphy_sleep_env` should be viewable to juypter (inside this notebook!)
5. On the top bar, select `Kernel`, then `Change Kernel`and select `anphy_sleep_env`

Make sure you change the Kenel before starting to run this notebook. If you are missing any imports, use `pip install` to add them to the env. 


In [1]:
# If ModuleNotFound, follow steps 1 - 3.c from above, and then run pip install <module name> from terminal

import mne
import numpy
import pandas as pd
from pathlib import Path 
import torch

## Renaming files 

The files structure is as follows: 
- `patient_records`

  - `EPCTL1`
      - `.edf` which is the data sequences 
      - `.txt` files which is the labels 
   - `EPCTL2`
   - `EPCTL29`


We rename all `.edf` files to be `EPCTL11<id_num>_data.edf`. 
Likewise, we rename all `.txt` file to be `EPCTL11<id_num>_label.txt`. 

In [2]:
from pathlib import Path

parent_dir = Path("patient_records/raw")
        
for edf_file in parent_dir.rglob("*.edf"):
    new_name = f"{edf_file.parent.name}_data.edf"
    edf_file.rename(edf_file.with_name(new_name))
    
for txt_file in parent_dir.rglob("*.txt"):
    new_name = f"{txt_file.parent.name}_labels.txt"
    txt_file.rename(txt_file.with_name(new_name))

# Data Preprocessing Overview
---

## 1. Channel Name Cleaning

Clean channel names by removing suffixes like "-Ref" or other hyphenated additions, while preserving special channels such as RLEG and LLEG. This ensures consistent naming across all EDF files and avoids errors when selecting channels for preprocessing.

## 2. Channel Selection

We begin by selecting a consistent subset of EEG channels and excluding unwanted channels.  

This ensures:
- Consistent feature dimensions across all patients
- Removal of noisy or irrelevant signals

---

## 3. Resampling

The original recordings are sampled at **1000 Hz**.  
We downsample to `resample_sfreq` (currently **100 Hz** but can be changed) to reduce memory usage and computational cost.

For example, if `resample_sfreq = 100 Hz`, each 30-second sleep epoch contains:

30 × 100 = **3000 time steps** instead of **30,000 time steps**

This reduces model input size while preserving relevant EEG dynamics. 

---

## 4. Epoch Creation (30-Second Windows)

We segment each recording into fixed-length 30-second windows using MNE, this that is how the data is laid out.

### Small Implementation Detail

MNE treats `tmax` as inclusive.  
If `tmax=30`, this would produce 3001 samples instead of 3000.

To ensure exactly 3000 samples per epoch, we use:

`tmax = 30 - 1/sfreq`

---

## 5. Handling Partial Final Epochs

If the recording duration is not perfectly divisible by 30 seconds, the final segment may be shorter.

MNE automatically drops incomplete epochs.

We intentionally **do not the last pad partial epochs**, to avoid introducing artificial data.  
Instead, we truncate the labels to match the number of valid epochs.

---

## 6. Tensor Formatting for CNN

MNE outputs data as:

(batch_size, channels, time_steps)

CNN expect:

(batch_size, time_steps, features)

So, we are good to leave the data as is.

---

## 7. Label Processing

Sleep stage labels are read from `.txt` files and mapped from string labels (e.g., W, N1, N2, etc.) to integer class IDs.

We ensure the number of labels matches the number of valid epochs by truncating to the minimum length.

---

## 8. Saving Data

Data is saved as a torch `.pt` binary file inside of `clean` folder. 

---

## Final Output Per Patient

For each recording:

- **X** → EEG tensor of shape  
  `(num_epochs, 30 * resample_sfreq, num_channels)`

- **y** → Sleep stage labels of shape  
  `(num_epochs,)`

These tensors are ready for training an LSTM-based sleep stage classifier.

In [3]:
# --------------------------------
# ---------DATA CONTROLS----------
# --------------------------------

CHANNELS = ['Fp1', 'Fp2', 'F3', 'F4', 'C3', 'C4', 'P3', 'P4', 
            'O1', 'O2', 'F7', 'F8', 'T3', 'T4', 'T5', 'T6', 
            'FZ', 'CZ', 'PZ', 'SO1', 'SO2', 'F9', 'F10', 'ZY1', 'ZY2', 
            'T9', 'T10', 'P9', 'P10', 'AF7', 'AF3', 'F11', 'F5', 'F1', 
            'FT11', 'FT9', 'FT7', 'FC5', 'FC3', 'FC1', 'FCZ', 'C5', 'C1', 
            'TP11', 'TP9', 'TP7', 'CP3', 'CP1', 'P11', 'P5', 'P1', 'PO7', 
            'PO3', 'POZ', 'OZ', 'FPZ', 'AFZ', 'AF4', 'AF8', 'F2', 'F6', 'F12', 
            'FC2', 'FC4', 'FC6', 'FT8', 'FT10', 'FT12', 'C6', 'C2', 'CPZ', 
            'CP2', 'CP4', 'CP6', 'TP8', 'TP10', 'TP12', 'P2', 'P6', 'P12', 
            'PO4', 'PO8', 'ChEMG1', 'ChEMG2', 'RLEG-', 'RLEG+', 'LLEG-', 'LLEG+', 
            'EOG2', 'EOG1', 'ECG2', 'ECG1', 'CP5'] # Constant for all channels present
excluded_channels = [] # put channels you want to NOT include here
selected_channels = [channel for channel in CHANNELS if channel not in excluded_channels]

resample_sfreq = 100 # we use this to downsample the Hz, cause original sample frequency is 1000.0 Hz

# --------------------------------
# ---------LABEL CONTROLS---------
# --------------------------------
SLEEP_STAGE_MAP = {'L':0, 'W':1, 'N1':2, 'N2':3, 'N3':4, 'R':5}


# --------------------------------
# --------PROCESSSING LOOP--------
# --------------------------------
parent_dir = Path("patient_records")
raw_dir = parent_dir.joinpath("raw")
clean_dir = parent_dir.joinpath("clean")
raw_dir.mkdir(parents=True, exist_ok=True)
clean_dir.mkdir(parents=True, exist_ok=True)
edf_files = sorted(parent_dir.rglob("*.edf"))
txt_files = sorted(parent_dir.rglob("*.txt"))

for data_file, label_file in zip(edf_files, txt_files):
    # Loads the file in 
    # Preload = True would move the entire file into RAM...PACE/ICE might be able to handle
    # Kernel might crash though...
    raw = mne.io.read_raw_edf(data_file, preload=False)
    
    # We clean the channel names due to some inconsistencies. Some contain "-" or "-Ref"
    remapping = {channel_name : channel_name.split("-")[0] for channel_name in raw.info["ch_names"] 
                 if "-" in channel_name 
                 and "RLEG" not in channel_name 
                 and "LLEG" not in channel_name}
    raw.rename_channels(remapping)

    # Picks the channels we selected. This is an inplace operation. WE OPTED TO DO THIS LATER BEOFRE LSTM TRAINING
#     raw.pick(selected_channels)
#     print(raw.get_data().shape)

    # For each 30-second time slice, we have 93 channels with each channel having samples due to the 1000.0 Hz
    # We resample per channel, to only have resample_sfreq samples, MASSIVE input size reduction
    raw.resample(resample_sfreq)

    # Make a set of events separated by a fixed duration, whichi in our case is 30-second per epoch
    events = mne.make_fixed_length_events(raw, duration=30)


    # IMPORTANT:
    # In MNE, epochs are defined as inclusive of both tmin and tmax.
    # That means the number of samples is:
    #     n_samples = (tmax - tmin) * sfreq + 1
    #
    # If we set tmax=30 for a 30-second epoch at 100 Hz,
    # we would get (30 * 100) + 1 = 3001 samples,
    # which is slightly longer than 30 seconds.
    #
    # To obtain exactly (epoch_duration * sfreq) samples
    # (e.g., 30 * 100 = 3000 samples),
    # we subtract one sample duration:
    #     tmax = epoch_duration - 1/sfreq
    epochs = mne.Epochs(
        raw,
        events,
        tmin=0,
        tmax=30 - 1/raw.info["sfreq"], # this is a small thing needed to ensure tmax is 30, not 30.000001 
        baseline=None,
        preload=False
    )

    # This is going to output (batch_size, num_features, time_steps), which works for CNNs
    # batch_size -> this is how many 30-second samples we have for a patient 
    # num_features -> the number of channels we are using
    # time_steps -> the sampling sfreq × seconds_per_epoch
    print("Epochs shape:", epochs.get_data().shape)

    # We read in the text labels
    label_df = pd.read_csv(label_file, sep='\t', header=None)

    # Name each column for easier processing
    label_df.columns = ["Sleep Stage", "Time (sec)", "Interval Lenght (sec)"]
    print(label_df.head())

    # We remap the sleep stage to convert from words to numerical representation
    label_df["Sleep Stage"] = label_df["Sleep Stage"].map(SLEEP_STAGE_MAP)

    # convert to numpy array
    y = label_df["Sleep Stage"].to_numpy()
    X = epochs.get_data()

    # In the data, sometimes the final epoch is < 30 second resulting in the time_step 
    # dimension size being smaller that all the other entires.
    # To prevent adding padding in the time_steps dim, and thus artificial data, 
    # we just truncate the last epoch if necessary
    min_len = min(len(epochs), len(label_df))
    X = X[:min_len]
    y = y[:min_len]

    # Saving the data. We save it as one file.
    save_path = clean_dir / f"{data_file.parent.name}_dataset.pt"
    torch.save({ "X": X, "y": y}, save_path)

Extracting EDF parameters from patient_records/raw/EPCTL01/EPCTL01_data.edf...
Setting channel info structure...
Creating raw.info structure...
Not setting metadata
957 matching events found
No baseline correction applied
0 projection items activated
Using data from preloaded Raw for 957 events and 3000 original time points ...
0 bad epochs dropped
Epochs shape: (957, 93, 3000)
  Sleep Stage  Time (sec)  Interval Lenght (sec)
0           W           0                     30
1           W          30                     30
2          N1          60                     30
3          N1          90                     30
4          N1         120                     30
Using data from preloaded Raw for 957 events and 3000 original time points ...
Extracting EDF parameters from patient_records/raw/EPCTL02/EPCTL02_data.edf...
Setting channel info structure...
Creating raw.info structure...
Not setting metadata
1072 matching events found
No baseline correction applied
0 projection items acti

Using data from preloaded Raw for 791 events and 3000 original time points ...
Extracting EDF parameters from patient_records/raw/EPCTL12/EPCTL12_data.edf...
Setting channel info structure...
Creating raw.info structure...
Not setting metadata
782 matching events found
No baseline correction applied
0 projection items activated
Using data from preloaded Raw for 782 events and 3000 original time points ...
0 bad epochs dropped
Epochs shape: (782, 93, 3000)
  Sleep Stage  Time (sec)  Interval Lenght (sec)
0           L           0                     30
1           L          30                     30
2           L          60                     30
3           L          90                     30
4           L         120                     30
Using data from preloaded Raw for 782 events and 3000 original time points ...
Extracting EDF parameters from patient_records/raw/EPCTL13/EPCTL13_data.edf...
Setting channel info structure...
Creating raw.info structure...
Not setting metadata
92

Extracting EDF parameters from patient_records/raw/EPCTL23/EPCTL23_data.edf...
Setting channel info structure...
Creating raw.info structure...
Not setting metadata
957 matching events found
No baseline correction applied
0 projection items activated
Using data from preloaded Raw for 957 events and 3000 original time points ...
0 bad epochs dropped
Epochs shape: (957, 93, 3000)
  Sleep Stage  Time (sec)  Interval Lenght (sec)
0           L          30                     30
1           L          60                     30
2           L          90                     30
3           L         120                     30
4           L         150                     30
Using data from preloaded Raw for 957 events and 3000 original time points ...
Extracting EDF parameters from patient_records/raw/EPCTL24/EPCTL24_data.edf...
Setting channel info structure...
Creating raw.info structure...
Not setting metadata
979 matching events found
No baseline correction applied
0 projection items activ

## Loading Data

When loading the data files, it should be done like this. Make sure to set `weights_only=False`.


In [4]:
data = torch.load(clean_dir / f"EPCTL01_dataset.pt",  weights_only=False)
X = data["X"]
y = data["y"]
print(X.shape)
print(y.shape)

(957, 93, 3000)
(957,)
